<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 처음부터 구현하는 Llama 3.2 (독립 실행형 노트북)

- 이 노트북은 의도적으로 최소화되었으며 Llama 3.2 1B 및 3B LLM을 구현하는 코드에 집중합니다
- 개별 구성 요소와 GPT, Llama 2, Llama 3 간의 관계를 설명하는 단계별 가이드는 다음 동반 노트북을 참조하세요:
  - [처음부터 구현한 GPT 아키텍처를 Llama 2로 변환하기](converting-gpt-to-llama2.ipynb)
  - [처음부터 구현하는 Llama 2를 Llama 3.2로 변환하기](converting-llama2-to-llama3.ipynb)
  

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/llama32.webp" width="700px">
  
  
- 코드에 대한 정보:
  - 모든 코드는 제가 직접 작성한 코드로, [Build A Large Language Model (From Scratch)](http://mng.bz/orYv) 책에서 구현한 모델 코드에 Llama 3 아키텍처를 매핑한 것입니다; 코드는 허용적인 오픈소스 Apache 2.0 라이선스 하에 공개됩니다 ([LICENSE.txt](https://github.com/rasbt/LLMs-from-scratch/blob/main/LICENSE.txt) 참조)
  - 토크나이저 코드는 Meta AI가 Tiktoken GPT-4 토크나이저를 확장하는 데 사용한 원본 [Llama 3 토크나이저 코드](https://github.com/meta-llama/llama3/blob/main/llama/tokenizer.py)에서 영감을 받았습니다
  - RoPE 재조정 섹션은 `transformers` 라이브러리의 [_compute_llama3_parameters 함수](https://github.com/huggingface/transformers/blob/5c1027bf09717f664b579e01cbb8ec3ef5aeb140/src/transformers/modeling_rope_utils.py#L329-L347)에서 영감을 받았습니다

In [ ]:
# pip install -r https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch05/07_gpt_to_llama/requirements-extra.txt

In [ ]:
from importlib.metadata import version

pkgs = [
    "blobfile",         # 사전 훈련된 가중치 다운로드용
    "huggingface_hub",  # 사전 훈련된 가중치 다운로드용
    "tiktoken",         # 토크나이저 구현용
    "torch",            # 모델 구현용
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

&nbsp;
# 1. 아키텍처 코드

In [ ]:
import torch
import torch.nn as nn


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.silu(x_fc1) * x_fc2
        return self.fc3(x)

In [ ]:
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, freq_config=None, dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # 역주파수 계산
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))

    # 주파수 조정
    if freq_config is not None:
        low_freq_wavelen = freq_config["original_context_length"] / freq_config["low_freq_factor"]
        high_freq_wavelen = freq_config["original_context_length"] / freq_config["high_freq_factor"]

        wavelen = 2 * torch.pi / inv_freq

        inv_freq_llama = torch.where(
            wavelen > low_freq_wavelen, inv_freq / freq_config["factor"], inv_freq
        )

        smooth_factor = (freq_config["original_context_length"] / wavelen - freq_config["low_freq_factor"]) / (
            freq_config["high_freq_factor"] - freq_config["low_freq_factor"]
        )

        smoothed_inv_freq = (
            (1 - smooth_factor) * (inv_freq / freq_config["factor"]) + smooth_factor * inv_freq
        )

        is_medium_freq = (wavelen <= low_freq_wavelen) & (wavelen >= high_freq_wavelen)
        inv_freq_llama = torch.where(is_medium_freq, smoothed_inv_freq, inv_freq_llama)
        inv_freq = inv_freq_llama

    # 위치 인덱스 생성
    positions = torch.arange(context_length, dtype=dtype)

    # 각도 계산
    angles = positions[:, None] * inv_freq[None, :]  # 모양: (context_length, head_dim // 2)

    # head_dim에 맞게 각도 확장
    angles = torch.cat([angles, angles], dim=1)  # 모양: (context_length, head_dim)

    # 사인과 코사인 미리 계산
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


def apply_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # x를 첫 번째 반과 두 번째 반으로 분할
    x1 = x[..., : head_dim // 2]  # 첫 번째 반
    x2 = x[..., head_dim // 2 :]  # 두 번째 반

    # sin과 cos 모양 조정
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # 모양: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # 회전 변환 적용
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    # cos와 sin 회전을 적용한 후 낮은 정밀도를 사용해도 괜찮습니다
    return x_rotated.to(dtype=x.dtype)

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(
            self, d_in, d_out, num_heads,
            num_kv_groups,
            dtype=None
        ):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_key = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * self.head_dim, bias=False, dtype=dtype)
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        self.W_query = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.out_proj = nn.Linear(d_out, d_out, bias=False, dtype=dtype)

    def forward(self, x, mask, cos, sin):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)  # 모양: (b, num_tokens, d_out)
        keys = self.W_key(x)  # 모양: (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)  # 모양: (b, num_tokens, num_kv_groups * head_dim)

        # 쿼리, 키, 값 재구성
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim)

        # 키, 값, 쿼리 전치
        keys = keys.transpose(1, 2)  # 모양: (b, num_kv_groups, num_tokens, head_dim)
        values = values.transpose(1, 2)  # 모양: (b, num_kv_groups, num_tokens, head_dim)
        queries = queries.transpose(1, 2)  # 모양: (b, num_heads, num_tokens, head_dim)

        # RoPE 적용
        keys = apply_rope(keys, cos, sin)
        queries = apply_rope(queries, cos, sin)

        # 헤드 수에 맞게 키와 값 확장
        # 모양: (b, num_heads, num_tokens, head_dim)
        keys = keys.repeat_interleave(self.group_size, dim=1)  # 모양: (b, num_heads, num_tokens, head_dim)
        values = values.repeat_interleave(self.group_size, dim=1)  # 모양: (b, num_heads, num_tokens, head_dim)
        # 예를 들어, dim=1(쿼리 그룹)을 따라 repeat_interleave 전:
        #   [K1, K2]
        # repeat_interleave 후 (각 쿼리 그룹이 group_size만큼 반복됨):
        #   [K1, K1, K2, K2]
        # 대신 일반적인 repeat을 사용했다면 다음과 같이 됩니다:
        #   [K1, K2, K1, K2]

        # 인과 마스크를 사용한 스케일드 닷-프로덕트 어텐션(즉, 셀프 어텐션) 계산
        # 모양: (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)  # 각 헤드에 대한 내적

        # 어텐션 스코어 계산
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        assert keys.shape[-1] == self.head_dim

        # 모양: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # 헤드 결합, 여기서 self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # 선택적 프로젝션

        return context_vec

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            num_kv_groups=cfg["n_kv_groups"],
            dtype=cfg["dtype"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = nn.RMSNorm(cfg["emb_dim"], eps=1e-5, dtype=cfg["dtype"])
        self.norm2 = nn.RMSNorm(cfg["emb_dim"], eps=1e-5, dtype=cfg["dtype"])

    def forward(self, x, mask, cos, sin):
        # 어텐션 블록에 대한 바로가기 연결
        shortcut = x
        x = self.norm1(x)
        x = self.att(x, mask, cos, sin)  # 모양 [batch_size, num_tokens, emb_size]
        x = x + shortcut  # 원본 입력 다시 추가

        # 피드포워드 블록에 대한 바로가기 연결
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut  # 원본 입력 다시 추가

        return x

In [ ]:
class Llama3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # 메인 모델 매개변수
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        self.trf_blocks = nn.ModuleList(  # Sequential은 하나의 입력만 받을 수 있고, 우리는 `x, mask, cos, sin`이 필요하므로 ModuleList 사용
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = nn.RMSNorm(cfg["emb_dim"], eps=1e-5, dtype=cfg["dtype"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        # 재사용 가능한 유틸리티
        cos, sin = compute_rope_params(
            head_dim=cfg["emb_dim"] // cfg["n_heads"],
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"],
            freq_config=cfg["rope_freq"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.cfg = cfg


    def forward(self, in_idx):
        # 순전파
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds

        num_tokens = x.shape[1]
        mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1)
        
        for block in self.trf_blocks:
            x = block(x, mask, self.cos, self.sin)
        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

&nbsp;
# 2. 모델 초기화

- 이 노트북의 나머지 부분은 Llama 3.2 1B 모델을 사용합니다. 3B 모델 변형을 사용하려면 다음 코드 셀에서 두 번째 구성 파일의 주석을 해제하면 됩니다

In [ ]:
# Llama 3.2 1B

LLAMA32_CONFIG = {
    "vocab_size": 128_256,           # 어휘 크기(vocabulary size)
    "context_length": 131_072,       # 모델 훈련에 사용된 컨텍스트 길이
    "emb_dim": 2048,                 # 임베딩 차원(embedding dimension)
    "n_heads": 32,                   # 어텐션 헤드 수(number of attention heads)
    "n_layers": 16,                  # 레이어 수(number of layers)
    "hidden_dim": 8192,              # FeedForward의 중간 차원 크기
    "n_kv_groups": 8,                # 그룹화된 쿼리 어텐션을 위한 키-값 그룹
    "rope_base": 500_000.0,          # RoPE의 "theta" 기본값
    "dtype": torch.bfloat16,         # 메모리 사용량을 줄이기 위한 낮은 정밀도 dtype
    "rope_freq": {                   # RoPE 주파수 스케일링
        "factor": 32.0,
        "low_freq_factor": 1.0,
        "high_freq_factor": 4.0,
        "original_context_length": 8192,
    }
}

# Llama 3.2 3B

# LLAMA32_CONFIG = {
#     "vocab_size": 128_256,           # 어휘 크기(vocabulary size)
#     "context_length": 131_072,       # 모델 훈련에 사용된 컨텍스트 길이
#     "emb_dim": 3072,                 # 임베딩 차원(embedding dimension)
#     "n_heads": 24,                   # 어텐션 헤드 수(number of attention heads)
#     "n_layers": 28,                  # 레이어 수(number of layers)
#     "hidden_dim": 8192,              # FeedForward의 중간 차원 크기
#     "n_kv_groups": 8,                # 그룹화된 쿼리 어텐션을 위한 키-값 그룹
#     "rope_base": 500_000.0,          # RoPE의 "theta" 기본값
#     "dtype": torch.bfloat16,         # 메모리 사용량을 줄이기 위한 낮은 정밀도 dtype
#     "rope_freq": {                   # RoPE 주파수 스케일링
#         "factor": 32.0,
#         "low_freq_factor": 1.0,
#         "high_freq_factor": 4.0,
#         "original_context_length": 8192,
#     }
# }

LLAMA_SIZE_STR = "1B" if LLAMA32_CONFIG["emb_dim"] == 2048 else "3B"

In [ ]:
model = Llama3Model(LLAMA32_CONFIG)

- 버퍼가 (낭비적으로) 재생성되는 대신 재사용됨을 확인하기 위해 다음이 True를 출력할 것으로 예상됩니다:

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# 가중치 묶기 고려
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

In [ ]:
def model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # 매개변수당 총 요소 수 계산
        param_size = param.numel()
        total_params += param_size
        # 이 매개변수에 대해 기울기가 저장되는지 확인
        if param.requires_grad:
            total_grads += param_size

    # 버퍼 크기 계산 (메모리가 필요한 비매개변수)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # 바이트 크기 = (요소 수) * (각 요소의 바이트 크기)
    # 매개변수와 기울기가 입력 dtype과 같은 유형으로 저장된다고 가정
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # 바이트를 기가바이트로 변환
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

print(f"float32 (PyTorch default): {model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device);

&nbsp;
# 3. 토크나이저 로드

In [ ]:
import os
from pathlib import Path

import tiktoken
from tiktoken.load import load_tiktoken_bpe



class Tokenizer:
    """Llama-3 특수 ID를 추적하는 tiktoken 주변의 얇은 래퍼."""
    def __init__(self, model_path):
        if not os.path.isfile(model_path):
            raise FileNotFoundError(model_path)

        mergeable = load_tiktoken_bpe(model_path)

        # Meta의 tokenizer.json에서 하드코딩
        self.special = {
            "<|begin_of_text|>": 128000,
            "<|end_of_text|>": 128001,
            "<|start_header_id|>": 128006,
            "<|end_header_id|>": 128007,
            "<|eot_id|>": 128009,
        }
        self.special.update({f"<|reserved_{i}|>": 128002 + i
                             for i in range(256)
                             if 128002 + i not in self.special.values()})

        self.model = tiktoken.Encoding(
            name=Path(model_path).name,
            pat_str=r"(?i:'s|'t|'re|'ve|'m|'ll|'d)"
                    r"|[^\r\n\p{L}\p{N}]?\p{L}+"
                    r"|\p{N}{1,3}"
                    r"| ?[^\s\p{L}\p{N}]+[\r\n]*"
                    r"|\s*[\r\n]+"
                    r"|\s+(?!\S)"
                    r"|\s+",
            mergeable_ranks=mergeable,
            special_tokens=self.special,
        )

    def encode(self, text, bos=False, eos=False):
        ids = ([self.special["<|begin_of_text|>"]] if bos else []) \
              + self.model.encode(text)
        if eos:
            ids.append(self.special["<|end_of_text|>"])
        return ids

    def decode(self, ids):
        return self.model.decode(ids)


class ChatFormat:

    def __init__(self, tokenizer: Tokenizer, *,
                 default_system="You are a helpful assistant."):
        self.tok = tokenizer
        self.default_system = default_system

    def _header(self, role):
        """<|start_header_id|>role<|end_header_id|>\n\n을 인코딩"""
        return (
            [self.tok.special["<|start_header_id|>"]]
            + self.tok.encode(role)
            + [self.tok.special["<|end_header_id|>"]]
            + self.tok.encode("\n\n")
        )

    def encode(self, user_message, system_message=None):
        sys_msg = system_message if system_message is not None else self.default_system

        ids = [self.tok.special["<|begin_of_text|>"]]

        # 시스템
        ids += self._header("system")
        ids += self.tok.encode(sys_msg)
        ids += [self.tok.special["<|eot_id|>"]]

        # 사용자
        ids += self._header("user")
        ids += self.tok.encode(user_message)
        ids += [self.tok.special["<|eot_id|>"]]

        # 어시스턴트 헤더 (아직 콘텐츠 없음)
        ids += self._header("assistant")

        return ids

- Meta AI는 파일을 다운로드하기 전에 Llama 3.2 라이선스 조건에 동의해야 합니다. 이를 위해서는 Hugging Face Hub 계정을 만들고 [meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B) 저장소를 방문하여 조건에 동의해야 합니다
- 다음으로 액세스 토큰을 만들어야 합니다. 읽기 권한이 있는 액세스 토큰을 생성하려면 우상단의 프로필 사진을 클릭하고 "Settings"를 클릭하세요


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/settings.webp?1" width="300px">

- 그런 다음 액세스 토큰을 만들고 복사하여 다음 코드 셀에 복사하여 붙여넣을 수 있습니다

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/access-token.webp?1" width="600px">

In [ ]:
# 노트북을 처음 실행하는 경우 다음 코드의 주석을 해제하고 실행하세요

# from huggingface_hub import login
# login()

In [ ]:
from huggingface_hub import hf_hub_download

tokenizer_file_path = hf_hub_download(
    repo_id=f"meta-llama/Llama-3.2-{LLAMA_SIZE_STR}-Instruct",
    filename="original/tokenizer.model",
    local_dir=f"Llama-3.2-{LLAMA_SIZE_STR}-Instruct"
)

In [ ]:
tokenizer = Tokenizer(tokenizer_file_path)
chat_tokenizer = ChatFormat(tokenizer)

&nbsp;
# 4. 사전 훈련된 가중치 로드

In [ ]:
def assign(left, right, tensor_name="unknown"):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}")

    if isinstance(right, torch.Tensor):
        return torch.nn.Parameter(right.clone().detach())
    else:
        return torch.nn.Parameter(torch.tensor(right))


def load_weights_into_llama(model, param_config, params):
    model.tok_emb.weight = assign(model.tok_emb.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")

    for l in range(param_config["n_layers"]):

        # 어텐션 가중치 로드
        model.trf_blocks[l].att.W_query.weight = assign(
            model.trf_blocks[l].att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight"
        )
        model.trf_blocks[l].att.W_key.weight = assign(
            model.trf_blocks[l].att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight"
        )
        model.trf_blocks[l].att.W_value.weight = assign(
            model.trf_blocks[l].att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight"
        )
        model.trf_blocks[l].att.out_proj.weight = assign(
            model.trf_blocks[l].att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight"
        )
        model.trf_blocks[l].norm1.weight = assign(
            model.trf_blocks[l].norm1.weight,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight"
        )

        # FeedForward 가중치 로드
        model.trf_blocks[l].ff.fc1.weight = assign(
            model.trf_blocks[l].ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight"
        )
        model.trf_blocks[l].ff.fc2.weight = assign(
            model.trf_blocks[l].ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight"
        )
        model.trf_blocks[l].ff.fc3.weight = assign(
            model.trf_blocks[l].ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight"
        )
        model.trf_blocks[l].norm2.weight = assign(
            model.trf_blocks[l].norm2.weight,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight"
        )

    # 출력 레이어 가중치 로드
    model.final_norm.weight = assign(model.final_norm.weight, params["model.norm.weight"], "model.norm.weight")

    if "lm_head.weight" in params.keys():
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    else:
        model.out_head.weight = assign(model.out_head.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")
        print("Model uses weight tying.")

In [ ]:
from safetensors.torch import load_file


if LLAMA_SIZE_STR == "1B":
    weights_file = hf_hub_download(
        repo_id=f"meta-llama/Llama-3.2-{LLAMA_SIZE_STR}-Instruct",
        filename="model.safetensors",
        local_dir=f"Llama-3.2-{LLAMA_SIZE_STR}-Instruct"
    )
    combined_weights = load_file(weights_file)


else:
    combined_weights = {}
    for i in range(1, 3):
        weights_file = hf_hub_download(
            repo_id=f"meta-llama/Llama-3.2-{LLAMA_SIZE_STR}-Instruct",
            filename=f"model-0000{i}-of-00002.safetensors",
            local_dir=f"Llama-3.2-{LLAMA_SIZE_STR}-Instruct"
        )
        current_weights = load_file(weights_file)
        combined_weights.update(current_weights)


load_weights_into_llama(model, LLAMA32_CONFIG, combined_weights)
model.to(device)
del combined_weights  # 메모리 해제

In [ ]:
print("Weight tying:", torch.equal(model.tok_emb.weight, model.out_head.weight))

&nbsp;
# 5. 텍스트 생성

In [ ]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # 배치 차원 추가
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)  # 배치 차원 제거
    return tokenizer.decode(flat.tolist())


def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # For-loop은 이전과 동일: 로짓을 얻고, 마지막 시간 단계에만 집중
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: top_k 샘플링으로 로짓 필터링
        if top_k is not None:
            # top_k 값만 유지
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float('-inf')).to(logits.device), logits)

        # New: 온도 스케일링 적용
        if temperature > 0.0:
            logits = logits / temperature

            # 확률을 얻기 위해 소프트맥스 적용
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # 분포에서 샘플링
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)

        # 그렇지 않으면 이전과 동일: 가장 높은 로짓 값을 가진 어휘 항목의 idx 가져오기
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)

        if idx_next == eos_id:  # 시퀀스 끝 토큰이 발견되고 eos_id가 지정된 경우 조기에 생성 중단
            break

        # 이전과 동일: 실행 중인 시퀀스에 샘플링된 인덱스 추가
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

In [ ]:
import time


PROMPT = "What do llamas eat?"

torch.manual_seed(123)

start = time.time()

token_ids = generate(
    model=model,
    idx=text_to_token_ids(PROMPT, chat_tokenizer).to(device),
    max_new_tokens=150,
    context_size=LLAMA32_CONFIG["context_length"],
    top_k=1,
    temperature=0.
)

print(f"Time: {time.time() - start:.2f} sec")

if torch.cuda.is_available():
    max_mem_bytes = torch.cuda.max_memory_allocated()
    max_mem_gb = max_mem_bytes / (1024 ** 3)
    print(f"Max memory allocated: {max_mem_gb:.2f} GB")

output_text = token_ids_to_text(token_ids, tokenizer)


def clean_text(text, header_end="assistant<|end_header_id|>\n\n"):
    # "<|end_header_id|>"의 첫 번째 발생 인덱스 찾기
    index = text.find(header_end)

    if index != -1:
        # "<|end_header_id|>" 이후부터 시작하는 부분 문자열 반환
        return text[index + len(header_end):].strip()  # Strip은 앞뒤 공백 제거
    else:
        # 토큰을 찾을 수 없으면 원본 텍스트 반환
        return text

print("\n\nOutput text:\n\n", clean_text(output_text))

&nbsp;
# 다음 단계는?

- 이 노트북은 의도적으로 최소화되었습니다. 개별 구성 요소에 대한 추가 설명에 관심이 있으시면 다음 두 동반 노트북을 확인하세요:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/gpt-and-all-llamas.webp">

  1. [처음부터 구현한 GPT 아키텍처를 Llama 2로 변환하기](converting-gpt-to-llama2.ipynb)
  2. [처음부터 구현하는 Llama 2를 Llama 3.2로 변환하기](converting-llama2-to-llama3.ipynb)
  
- 처음부터 대형 언어 모델을 구축하고 그 메커니즘에 대한 더 깊은 이해를 얻는 포괄적인 가이드에 관심이 있으시면 저의 [Build a Large Language Model (From Scratch)](http://mng.bz/orYv) 책을 좋아하실 것입니다

<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>